# Colon Histopathology: Tri-Branch CNN with Attention

This notebook trains a real binary classifier from the public LC25000 Kaggle dataset. It uses **only** `colon_n` and `colon_aca`:

- `colon_n` Ã¢â€ â€™ **Normal/Benign Colon Tissue** (class 0)
- `colon_aca` Ã¢â€ â€™ **Adenocarcinoma** (class 1)

The full tri-branch CNN and attention classifier are trained end-to-end. This is intentionally a separate notebook; the existing notebook is left unchanged. The model uses the preprocessing expected by this project's Flask backend.

## Before you start

Choose **GPU** in Colab (`Runtime Ã¢â€ â€™ Change runtime type Ã¢â€ â€™ T4 GPU`) or Kaggle (`Settings Ã¢â€ â€™ Accelerator Ã¢â€ â€™ GPU`, if available). On Kaggle, attach the public dataset using **Add Input**. On Colab, create a Kaggle API token at Kaggle **Account Ã¢â€ â€™ Settings Ã¢â€ â€™ Create New Token** and upload the downloaded `kaggle.json` when prompted. Keep that token private and never commit it.

Training and evaluation happen only when you run these cells. This notebook does not contain precomputed results. The output is an academic prototype, not a medical diagnostic tool.

## 1. Install and Import Required Libraries

This cell installs the small missing utilities and imports PyTorch, OpenCV, scikit-learn, and plotting/data libraries. Colab and Kaggle provide the PyTorch runtime; choose a GPU accelerator before continuing.

In [ ]:
%pip -q install kaggle opencv-python-headless scikit-learn

import json
import os
import random
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Not available; training will use CPU")

## 2. Download Dataset from Kaggle

On Kaggle, attach the public dataset with **Add Input**. On Colab, the next cell securely asks for your private Kaggle API token and downloads the dataset. The token is not displayed or included in the exported model.

In [ ]:
DATASET_SLUG = "andrewmvd/lung-and-colon-cancer-histopathological-images"
KAGGLE_INPUT = Path("/kaggle/input")

if KAGGLE_INPUT.exists():
    # In a Kaggle Notebook, first click Add Input and attach the public dataset.
    DATA_SEARCH_ROOT = KAGGLE_INPUT
    print("Kaggle environment found. Searching attached inputs:", KAGGLE_INPUT)
else:
    # In Colab, upload kaggle.json from Kaggle Account > Settings > Create New Token.
    from google.colab import files

    print("Upload the kaggle.json token file you downloaded from Kaggle.")
    uploaded_files = files.upload()
    if "kaggle.json" not in uploaded_files:
        raise FileNotFoundError("Upload the file named kaggle.json, then rerun this cell.")

    kaggle_config = Path.home() / ".kaggle"
    kaggle_config.mkdir(parents=True, exist_ok=True)
    token_path = kaggle_config / "kaggle.json"
    token_path.write_bytes(uploaded_files["kaggle.json"])
    token_path.chmod(0o600)

    download_dir = Path("/content/lc25000")
    download_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", DATASET_SLUG, "-p", str(download_dir), "--unzip"],
        check=True,
    )
    DATA_SEARCH_ROOT = download_dir
    print("Dataset downloaded and extracted to:", download_dir)

print("Dataset search root:", DATA_SEARCH_ROOT)

## 3. Extract and Organize Colon Images

## 4. Label Mapping and Data Preparation

The next cell finds only the dataset's `colon_n` and `colon_aca` folders, counts supported images, and builds the image-path/label table. All lung cancer folders are excluded.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def find_colon_folders(search_root):
    search_root = Path(search_root)
    for adenocarcinoma_dir in search_root.rglob("colon_aca"):
        if not adenocarcinoma_dir.is_dir():
            continue
        normal_dir = adenocarcinoma_dir.parent / "colon_n"
        if normal_dir.is_dir():
            return normal_dir, adenocarcinoma_dir
    return None, None

normal_dir, adenocarcinoma_dir = find_colon_folders(DATA_SEARCH_ROOT)
if normal_dir is None:
    raise FileNotFoundError(
        "Could not find sibling colon_n and colon_aca folders. "
        "In Kaggle, click Add Input and attach the Andrew MVD lung-and-colon dataset."
    )

print("Normal folder:", normal_dir)
print("Adenocarcinoma folder:", adenocarcinoma_dir)

CLASS_NAMES = ["Normal/Benign Colon Tissue", "Adenocarcinoma"]
CLASS_TO_IDX = {"colon_n": 0, "colon_aca": 1}
rows = []
for folder, label in [(normal_dir, CLASS_TO_IDX["colon_n"]), (adenocarcinoma_dir, CLASS_TO_IDX["colon_aca"])]:
    image_paths = sorted(path for path in folder.rglob("*") if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)
    rows.extend({"image_path": str(path), "label": label} for path in image_paths)

data = pd.DataFrame(rows)
if data.empty or set(data["label"].unique()) != {0, 1}:
    raise ValueError("Both colon_n and colon_aca must contain readable image files.")

print("Total selected colon images:", len(data))
print("Class counts (0=Normal/Benign, 1=Adenocarcinoma):")
print(data["label"].value_counts().sort_index())
print("\nThis notebook excludes all lung classes.")

## 5. Image Preprocessing and Augmentation

The shared preprocessing is deterministic at validation, test, and inference time. Training alone adds random horizontal and vertical flips.

In [ ]:
IMAGE_SIZE = 224
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def preprocess_rgb_array(rgb_image):
    """Apply the same deterministic preprocessing used by Flask inference."""
    lab_image = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2LAB)
    lightness, channel_a, channel_b = cv2.split(lab_image)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced_lightness = clahe.apply(lightness)
    enhanced_rgb = cv2.cvtColor(
        cv2.merge((enhanced_lightness, channel_a, channel_b)),
        cv2.COLOR_LAB2RGB,
    )

    enhanced_float = enhanced_rgb.astype(np.float32)
    minimum = float(enhanced_float.min())
    maximum = float(enhanced_float.max())
    if maximum > minimum:
        enhanced_float = (enhanced_float - minimum) / (maximum - minimum) * 255.0
    enhanced_uint8 = np.clip(enhanced_float, 0, 255).astype(np.uint8)
    resized = cv2.resize(enhanced_uint8, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)

    image_float = resized.astype(np.float32) / 255.0
    image_tensor = torch.from_numpy(image_float.transpose(2, 0, 1).copy())
    mean_tensor = torch.tensor(IMAGENET_MEAN)[:, None, None]
    std_tensor = torch.tensor(IMAGENET_STD)[:, None, None]
    return (image_tensor - mean_tensor) / std_tensor

class ColonImageDataset(Dataset):
    def __init__(self, dataframe, training=False):
        self.dataframe = dataframe.reset_index(drop=True)
        self.training = training
        self.augmentation = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
        ]) if training else None

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        with Image.open(row["image_path"]) as source:
            rgb_image = np.asarray(source.convert("RGB"))
        image_tensor = preprocess_rgb_array(rgb_image)
        if self.augmentation is not None:
            image_tensor = self.augmentation(image_tensor)
        label = torch.tensor(int(row["label"]), dtype=torch.long)
        return image_tensor, label

print("Deterministic preprocessing: RGB Ã¢â€ â€™ LAB CLAHE Ã¢â€ â€™ per-image min/max Ã¢â€ â€™ 224Ãƒâ€”224 Ã¢â€ â€™ ImageNet normalization")
print("Training augmentation: random horizontal/vertical flips only; validation and test are not augmented.")

## 6. Stratified Train/Validation/Test Split

This split reserves 70% for training, 15% for validation, and 15% for a final held-out test. Stratification preserves the class proportions in each split.

In [ ]:
train_df, temporary_df = train_test_split(
    data,
    test_size=0.30,
    random_state=SEED,
    stratify=data["label"],
)
validation_df, test_df = train_test_split(
    temporary_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temporary_df["label"],
)

for split_name, split_df in [("Train", train_df), ("Validation", validation_df), ("Test", test_df)]:
    print(f"{split_name:10s}: {len(split_df):5d} images | {split_df['label'].value_counts().sort_index().to_dict()}")

print("\nStratified split: approximately 70% train, 15% validation, 15% test.")

## 7. Build the Tri-Branch CNN and Attention Model

The model concatenates 128 fine-texture, 128 structural, and 128 global-context features in a fixed order. A learned attention gate weights the 384 features before the two-class head.

In [ ]:
class CNNBranch(nn.Module):
    def __init__(self, branch_type):
        super().__init__()
        if branch_type == "fine_texture":
            layers = [
                nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
                nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
                nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            ]
        elif branch_type == "structural":
            layers = [
                nn.Conv2d(3, 32, kernel_size=5, padding=2), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
                nn.Conv2d(32, 64, kernel_size=5, padding=2), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
                nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            ]
        elif branch_type == "global_context":
            layers = [
                nn.Conv2d(3, 32, kernel_size=3, padding=2, dilation=2), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
                nn.Conv2d(32, 64, kernel_size=3, padding=2, dilation=2), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
                nn.Conv2d(64, 128, kernel_size=3, padding=4, dilation=4), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            ]
        else:
            raise ValueError(f"Unknown branch type: {branch_type}")

        self.features = nn.Sequential(*layers)
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, image):
        return torch.flatten(self.global_pool(self.features(image)), start_dim=1)

class TriBranchAttentionNet(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.fine_texture_branch = CNNBranch("fine_texture")
        self.structural_branch = CNNBranch("structural")
        self.global_context_branch = CNNBranch("global_context")
        self.attention = nn.Sequential(
            nn.Linear(384, 96),
            nn.ReLU(inplace=True),
            nn.Linear(96, 384),
            nn.Sigmoid(),
        )
        self.classifier = nn.Sequential(
            nn.Linear(384, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.30),
            nn.Linear(128, num_classes),
        )

    def forward(self, image):
        fine = self.fine_texture_branch(image)
        structural = self.structural_branch(image)
        global_context = self.global_context_branch(image)
        fused = torch.cat((fine, structural, global_context), dim=1)
        attention_weights = self.attention(fused)
        refined = fused * attention_weights
        logits = self.classifier(refined)
        return logits

model = TriBranchAttentionNet(num_classes=len(CLASS_NAMES)).to(device)
print(model)
print("The CNN branches, attention block, and classifier head will all be trained together.")

## 8. Train the Model

Prepare the data loaders, then run end-to-end training with AdamW and early stopping. The best validation-loss checkpoint is restored before evaluation.

In [ ]:
BATCH_SIZE = 16
NUM_WORKERS = 2

train_dataset = ColonImageDataset(train_df, training=True)
validation_dataset = ColonImageDataset(validation_df, training=False)
test_dataset = ColonImageDataset(test_df, training=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print(f"Batches per epoch: train={len(train_loader)}, validation={len(validation_loader)}, test={len(test_loader)}")

In [ ]:
EPOCHS = 12
PATIENCE = 3
LEARNING_RATE = 1e-3

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
best_validation_loss = float("inf")
best_epoch = 0
best_state = None
epochs_without_improvement = 0
training_history = []

def run_epoch(data_loader, training):
    model.train(training)
    total_loss = 0.0
    all_predictions = []
    all_labels = []

    for images, labels in data_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * labels.size(0)
        all_predictions.extend(logits.argmax(dim=1).detach().cpu().tolist())
        all_labels.extend(labels.detach().cpu().tolist())

    average_loss = total_loss / len(data_loader.dataset)
    average_accuracy = accuracy_score(all_labels, all_predictions)
    return average_loss, average_accuracy

for epoch in range(1, EPOCHS + 1):
    started = time.time()
    train_loss, train_accuracy = run_epoch(train_loader, training=True)
    validation_loss, validation_accuracy = run_epoch(validation_loader, training=False)
    training_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "validation_loss": validation_loss,
        "validation_accuracy": validation_accuracy,
    })

    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        best_epoch = epoch
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    elapsed = time.time() - started
    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss {train_loss:.4f}, accuracy {train_accuracy:.3f} | "
        f"validation loss {validation_loss:.4f}, accuracy {validation_accuracy:.3f} | "
        f"{elapsed:.1f}s"
    )

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping after {PATIENCE} epochs without validation-loss improvement.")
        break

if best_state is None:
    raise RuntimeError("Training did not produce a best model checkpoint.")
model.load_state_dict(best_state)
model.eval()
print(f"Restored the measured best validation checkpoint from epoch {best_epoch}.")
print("Training is complete only if the epoch output above completed without errors.")

## 9. Evaluate on the Held-Out Test Set

The next cell calculates accuracy, per-class precision/recall/F1, and a confusion matrix from the untouched test split. These values appear only after you run the notebook.

In [ ]:
model.eval()
test_predictions = []
test_targets = []
with torch.no_grad():
    for images, labels in test_loader:
        logits = model(images.to(device))
        test_predictions.extend(logits.argmax(dim=1).cpu().tolist())
        test_targets.extend(labels.tolist())

test_accuracy = accuracy_score(test_targets, test_predictions)
print(f"Held-out test accuracy (measured): {test_accuracy:.4f}")
print("\nHeld-out test classification report:")
print(classification_report(
    test_targets,
    test_predictions,
    labels=[0, 1],
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0,
))

matrix = confusion_matrix(test_targets, test_predictions, labels=[0, 1])
ConfusionMatrixDisplay(matrix, display_labels=CLASS_NAMES).plot(cmap="Blues", values_format="d", xticks_rotation=20)
plt.title("Held-out test confusion matrix")
plt.tight_layout()
plt.show()
print("These metrics are produced by your run; no performance value is pre-filled in this notebook.")

## 10. Save the Trained Model and Class Labels

This saves only the checkpoint created after the training loop and held-out evaluation have run. It includes the trained weights, exact class order, label mapping, preprocessing configuration, and measured run metadata.

In [ ]:
if Path("/kaggle/working").exists():
    EXPORT_DIR = Path("/kaggle/working")
elif Path("/content").exists():
    EXPORT_DIR = Path("/content/colon_model_export")
else:
    EXPORT_DIR = Path.cwd() / "colon_model_export"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_path = EXPORT_DIR / "colon_tri_branch_attention.pth"
labels_path = EXPORT_DIR / "class_names.json"

checkpoint = {
    "schema_version": 1,
    "architecture": "TriBranchAttentionNet-v1",
    "model_state_dict": model.state_dict(),
    "class_names": CLASS_NAMES,
    "class_to_idx": CLASS_TO_IDX,
    "preprocessing": {
        "image_size": IMAGE_SIZE,
        "color_order": "RGB",
        "mclahe_clip_limit": 2.0,
        "mclahe_tile_grid_size": [8, 8],
        "per_image_min_max": True,
        "resize_interpolation": "INTER_AREA",
        "imagenet_mean": IMAGENET_MEAN.tolist(),
        "imagenet_std": IMAGENET_STD.tolist(),
        "training_augmentation": ["random horizontal flip", "random vertical flip"],
        "inference_augmentation": None,
    },
    "classifier_name": "Tri-Branch CNN + Feature Attention",
    "trained": True,
    "best_epoch": int(best_epoch),
    "best_validation_loss": float(best_validation_loss),
    "measured_test_accuracy": float(test_accuracy),
    "confidence_calibrated": False,
}
torch.save(checkpoint, checkpoint_path)
labels_path.write_text(
    json.dumps({"class_names": CLASS_NAMES, "class_to_idx": CLASS_TO_IDX}, indent=2),
    encoding="utf-8",
)

bundle_zip = EXPORT_DIR / "colon_model_bundle.zip"
with zipfile.ZipFile(bundle_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(checkpoint_path, arcname=checkpoint_path.name)
    archive.write(labels_path, arcname=labels_path.name)

print("Exported trained checkpoint:", checkpoint_path)
print("Exported class labels:", labels_path)
print("Bundle for download:", bundle_zip)

try:
    from google.colab import files as colab_files
    colab_files.download(str(bundle_zip))
except ImportError:
    print("On Kaggle, open the notebook Output panel and download colon_model_bundle.zip.")

print("Only a checkpoint produced after completing training and test evaluation is saved here.")

## 11. Download the Model for Flask

### Google Colab

1. In Colab, choose **Runtime â†’ Change runtime type â†’ T4 GPU** if available.
2. Use **Runtime â†’ Run all**. When the Kaggle download cell opens a file picker, choose your private `kaggle.json` token file.
3. Wait until training prints its epoch metrics, test evaluation, sample predictions, and checkpoint-reload confirmation. This notebook does not train until you run it.
4. Colab downloads `colon_model_bundle.zip`. Extract it on your computer.

### Kaggle Notebook

1. Create a Kaggle Notebook and click **Add Input** to attach the public lung-and-colon dataset.
2. Click **Run All** and wait for training, held-out test results, and checkpoint reload to finish.
3. Open the notebook **Output** panel and download `colon_model_bundle.zip`; extract it on your computer.

### Copy the two model files

1. In VS Code, open this project and create/open `backend/models/`.
2. Copy `colon_tri_branch_attention.pth` and `class_names.json` from the extracted bundle into `backend/models/`.
3. In the Flask terminal, make sure the project's Python 3.12 virtual environment is active, then run `python -m pip install -r backend\\requirements-model.txt` once to install PyTorch, NumPy, and OpenCV.
4. Restart Flask. Open `http://127.0.0.1:5001/api/health`; it should report `model.ready: true`.
5. Sign in, upload one JPG/PNG colon image, and click **Analyze image**. Try a `colon_n` example and a `colon_aca` example. History stores successful prediction metadata only; uploaded image bytes are not retained.

Raw softmax output is not calibrated, so the website does not display it as a confidence percentage. Test metrics and sample predictions are generated by your run; do not describe them as clinical performance.

## 12. Test Predictions on Real Samples from Both Classes

The next cells reload the saved weights and run one held-out image from each class with the exact inference preprocessing. Raw model scores are deliberately not labeled as confidence.

In [ ]:
model.eval()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for axis, true_label in zip(axes, [0, 1]):
    sample = test_df[test_df["label"] == true_label].sample(n=1, random_state=SEED).iloc[0]
    with Image.open(sample["image_path"]) as source:
        rgb_image = np.asarray(source.convert("RGB"))
    image_tensor = preprocess_rgb_array(rgb_image).unsqueeze(0).to(device)
    with torch.no_grad():
        predicted_label = int(model(image_tensor).argmax(dim=1).item())

    axis.imshow(rgb_image)
    axis.axis("off")
    axis.set_title(
        f"True: {CLASS_NAMES[true_label]}\nPredicted: {CLASS_NAMES[predicted_label]}",
        fontsize=9,
    )
plt.tight_layout()
plt.show()
print("These are actual predictions for held-out files; they are not guaranteed to be correct.")
print("No confidence percentage is reported: raw softmax scores are not calibrated probabilities.")

In [ ]:
# Reload the files exactly as Flask will, then run one real test image from each class.
loaded_checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
assert loaded_checkpoint["class_names"] == CLASS_NAMES
assert loaded_checkpoint["architecture"] == "TriBranchAttentionNet-v1"

reloaded_model = TriBranchAttentionNet(num_classes=len(loaded_checkpoint["class_names"])).to(device)
reloaded_model.load_state_dict(loaded_checkpoint["model_state_dict"], strict=True)
reloaded_model.eval()

for true_label, label_name in enumerate(CLASS_NAMES):
    sample = test_df[test_df["label"] == true_label].iloc[0]
    with Image.open(sample["image_path"]) as source:
        rgb_image = np.asarray(source.convert("RGB"))
    image_tensor = preprocess_rgb_array(rgb_image).unsqueeze(0).to(device)
    with torch.no_grad():
        predicted_label = int(reloaded_model(image_tensor).argmax(dim=1).item())
    print(f"True: {label_name} | Reloaded model prediction: {CLASS_NAMES[predicted_label]}")

print("Checkpoint reload and both real class paths passed. Predictions may still be wrong; use the measured test metrics above.")